# Quickstart

Profile an orders table, retrieve a finding, inspect its evidence, and check
what work supports it. This fixture contains a primary key, a region
lookup, a nearly unique external reference with two repeated values, and a
derived quantity. The known structure makes the result easy to check.

The basic workflow is `report = onceover.profile(df)`, followed by
`report.select(...)` for complete findings and `report.coverage` for the
scope of the search. The [capability guide](capabilities.ipynb) introduces
the other questions onceover can answer.


In [ ]:
import json
import random

import polars as pl

import onceover

rng = random.Random(7)
n = 2000

region_id = [rng.randrange(3) for _ in range(n)]
region_name = [["emea", "amer", "apac"][r] for r in region_id]
order_id = list(range(n))
external_ref = [f"R{i:05}" for i in range(n)]
external_ref[-1] = external_ref[0]   # two duplicated values
external_ref[-2] = external_ref[1]
quantity = [rng.randrange(1, 11) for _ in range(n)]
double_quantity = [2 * q for q in quantity]

df = pl.DataFrame({
    "order_id": order_id,
    "external_ref": external_ref,
    "region_id": region_id,
    "region_name": region_name,
    "quantity": quantity,
    "double_quantity": double_quantity,
})
from great_tables import GT, loc, style
(GT(df.head(4))
 .tab_header(title="Four rows from the orders fixture")
 .tab_options(table_width="100%", table_layout="fixed")
 .tab_style(style.css("white-space: normal; overflow-wrap: anywhere;"), loc.body()))

## Profile it

One call runs the detectors enabled by the default budget.
[](large_tables.ipynb) describes runtime and coverage on larger tables.

In [ ]:
report = onceover.profile(df)
print(report.overview["headline"])

The headline reports a key candidate, functional dependencies,
predictable columns, and conditional rules for this fixture.

## Inspect findings

`report.findings` is sorted by its priority score, `interestingness`:


In [ ]:
finding_index = report.findings_frame().select("category", "type", "claim").head(8)
(GT(finding_index)
 .tab_header(title="Highest-ranked findings", subtitle="Complete claims; retrieve a finding for all its evidence")
 .tab_options(table_width="100%", table_layout="fixed", table_font_size="14px")
 .tab_style(style.css("white-space: normal; overflow-wrap: anywhere; line-height: 1.5;"), loc.body()))

The report verifies mappings between `quantity` and `double_quantity`,
checks their affine formula, and finds the region lookup. It also reports
that `order_id` is unique while `external_ref` has two repeated values.

The following count shows the categories present in this report:


In [ ]:
from collections import Counter

print(Counter((f["category"], f.get("type")) for f in report.findings))

To inspect one column, use `report.select(column=...)`. The returned
entries are complete findings, in report order:

In [ ]:
for finding in report.select(column="external_ref"):
    print(f"[{finding['category']}] {finding['claim']}")

`external_ref` looks like an identifier, but it contains two repeated
values. Joining on it as though it were unique can produce extra rows.
Inspect the uniqueness finding before choosing it as a key.

## Inspect a complete finding


In [ ]:
finding = report.select(category="uniqueness", column="external_ref")[0]
print(json.dumps(finding, indent=2))

The result names its columns in `subject`. `specification` describes the
comparison, mapping, or formula, and `evidence` contains the evaluated row
counts and metrics. The [findings chapter](findings.ipynb) explains these
fields and shows how to filter on them.


In [ ]:
identity = next(f for f in report.select(category="dependencies", type="functional")
                if f["specification"].get("form") == "affine")
print(identity["claim"])
print("category:", identity["category"], "| type:", identity.get("type"))
print("form:", identity["specification"].get("form"))
ev = identity["evidence"][0]
print("basis:", ev["basis"], "| sampling:", ev["scope"]["sampling"])
print("validation:", ev["validation"]["method"])
print("metrics:", ev["metrics"])

The identity check fitted `y = a*x + b` on a sample, then checked the
residuals on all 2,000 rows. Every row matched within the stated tolerance.

## Inspect column summaries

`report.columns` contains counts, statistics, and suggested roles, keyed
by column name. For example:


In [ ]:
print(json.dumps(report.columns["region_id"], indent=2))
print(json.dumps(report.columns["double_quantity"]["numeric_stats"], indent=2))

`region_id` has three distinct numeric values and receives a categorical
role with a `treat_as_categorical` suggestion. `double_quantity` includes
exact distribution summaries. These counts use every input row; see
[Column summaries](columns.ipynb) for the available fields.

## Check which passes ran

`report.coverage` lists completed and skipped passes, along with their
sample sizes and limits:


In [ ]:
coverage_rows = [{"Pass": name, "Status": info["status"], "Detail": info["detail"]}
                 for name, info in report.coverage.items()]
(GT(pl.DataFrame(coverage_rows))
 .tab_header(title="What the profile examined")
 .tab_options(table_width="100%", table_layout="fixed", table_font_size="14px")
 .tab_style(style.css("white-space: normal; overflow-wrap: anywhere; line-height: 1.5;"), loc.body()))

`row_duplicates` can rule out duplicate rows using the verified unique,
null-free `order_id`. `functional_dependencies` reports its verification
sample. This fixture fits within that cap; larger inputs may be sampled.

## Declare what you know

A supplied calendar makes a gap check explicit. Here March is absent from
a monthly series that is too short for the inferred date detector. Hints
provide the expected cadence; the `expectations` evidence records
`reference_source="declared"`.


In [ ]:
import datetime

monthly = pl.DataFrame({
    "month": [datetime.date(2024, m, 1) for m in (1, 2, 4, 5, 6)],
    "revenue": [10.0, 11.0, 13.0, 12.0, 14.0],
})
hinted = onceover.profile(monthly, hints={
    "time": "month",
    "no_gaps": ["month"],
    "cadence": {"month": "monthly"},
})
for f in hinted.findings:
    if f["category"] == "gaps":
        print(f["claim"])
        print(f["evidence"][0]["metrics"])

## Save the report

`save()` writes the complete JSON report by default. For repeatable JSON,
use the same package version, data, and settings with no deadline and with
`timing=False`. A short digest records its omissions and links back to the
full report through finding IDs.


In [ ]:
import os
import tempfile

path = report.save(tempfile.mkdtemp() + "/profile.json")
print("saved", os.path.getsize(path), "bytes to", path)
print(report.to_digest(800))

`save()` writes JSON by default; `report.data` returns the parsed report.
`to_markdown()` produces a text summary suitable for sharing in an issue
or pull request.

## Further examples

- [Reading findings](findings.ipynb): fields, scores, and filtering.
- [Column summaries](columns.ipynb): roles and distribution statistics.
- [Coverage](coverage.ipynb): budgets, sampling, and detector selection.
- [Analysis notebook](https://github.com/fcela/onceover/blob/main/notebooks/onceover_analysis.ipynb):
  a larger fixture with charts and checks against planted relationships.


For the full method map, continue with [capabilities](capabilities.ipynb).
The [API reference](api.ipynb) covers targeting, hints, report comparison,
size-budgeted JSON, and per-pass timing.
